In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from Bio import SeqIO


pep_hits_long = pd.read_csv('long_090726.csv')

In [ ]:
lassa = pep_hits_long[pep_hits_long['species'] == 'Mammarenavirus lassaense'].copy()

## importing the reference sequences fetched from UniProt (UP000002473_11622.fasta)

In [11]:
records = list(SeqIO.parse('UP000002473_11622_LASV_Josiah_reference.fasta', 'fasta'))

for r in records:
    print(r.id, len(r.seq))

sp|O09705|L_LASSJ 2218
sp|O73557|Z_LASSJ 99
sp|P08669|GLYC_LASSJ 491
sp|P13699|NCAP_LASSJ 569


In [13]:
reference_by_protein = {
    "polymerase": str(next(r.seq for r in records if "O09705" in r.id)),
    "glycoprotein": str(next(r.seq for r in records if "P08669" in r.id)),
    "nucleoprotein": str(next(r.seq for r in records if "P13699" in r.id)),
    "z protein": str(next(r.seq for r in records if "O73557" in r.id)),
}

In [18]:
print(lassa['category'].value_counts())
print(lassa['sequence'].nunique())
print(lassa['protein_std'].unique())

category
Household contact    21052
Lassa survivor        9120
US healthy            6612
Name: count, dtype: int64
76
['polymerase' 'glycoprotein' 'nucleoprotein' 'z protein']


## import PairwiseAligner

In [26]:
from Bio.Align import PairwiseAligner

aligner = PairwiseAligner()
aligner.mode = 'local'

aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -5
aligner.extend_gap_score = -1 

In [27]:
def local_align_peptide(peptide, reference, aligner):
    alignment = aligner.align(reference, peptide)[0]

    return{
        'score': alignment.score, 
        'alignment': alignment
    }

## map to nucleoprotein

In [28]:
ref = reference_by_protein['nucleoprotein']

np = lassa[lassa['protein_std'].eq("nucleoprotein")].copy()

unique_np_peptides = np['sequence'].drop_duplicates()

In [30]:
unique_np_peptides

156    TERPLSSGVYMGNLSSQQLDQRRALLNMIGMAGGSQGNQPSRDGVV
157    ALLNMIGMAGGSQGNQPSRDGVVRVWDVKNADLLNNQFGTMPSLTL
242    KPGNTGSNKSLQSAGFAAGLTYSQLMTLKDSMLQLDPNAKTWIDIE
248    KVGTTGSNKSLQSAGFPAGLTYSQLMTLKDSMMQLDPSAKTWIDIE
249    KPGNNGSNRSLQSAGFPAGLTYSQLMTLKDSMLQLDPNAKTWMDIE
594    KAGSNGSNKSLQSAGFTAGLTYSQLMTLKDAMLQLDPNAKTWMDIE
605    KVGTAGSNKSLQSAGFPTGLTYSQLMTLKDSMMQLDPSAKTWIDIE
Name: sequence, dtype: object

In [36]:
np_result = local_align_peptide(unique_np_peptides[248], ref, aligner)

In [39]:
print(np_result['alignment'])

target          351 SSKSLQSAGFTAGLTYSQLMTLKDAMLQLDPNAKTWMDIE 391
                  0 |.||||||||.|||||||||||||.|.||||.||||.|||  40
query             6 SNKSLQSAGFPAGLTYSQLMTLKDSMMQLDPSAKTWIDIE  46

